# Queries, assumption audit, and bootstrap CIs

Beyond a single interventional probability: ATE, PN / PS / PNS, conditional do-queries, `audit_bound`, and `bound_ci`.


In [1]:
from partialid import (
    Assumptions,
    Query,
    Variables,
    audit_bound,
    bound,
    bound_ci,
)

V = Variables.from_binary("X", "Y")
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
a = Assumptions.order("X", "Y")


## Interventional, ATE, and probability-of-causation faces


In [2]:
queries = {
    "interventional": Query.interventional(outcome="Y", do={"X": 1}, value=1),
    "ATE": Query.ate(outcome="Y", treatment="X"),
    "PNS": Query.pns(),
    "PN": Query.pn(),
    "PS": Query.ps(),
}
for name, Q in queries.items():
    r = bound(V, P, Q, a)
    print(f"{name:14s}  {Q.describe()}")
    print(f"               [{r.lower:.4f}, {r.upper:.4f}]  width={r.width:.4f}  method={r.method}\n")


interventional  P(Y=1 | do(X=1))
               [0.3500, 0.8500]  width=0.5000  method=closed/manski_binary_xy

ATE             ATE: P(Y=1 | do(X=1)) - P(Y=1 | do(X=0))
               [-0.3500, 0.6500]  width=1.0000  method=fiber

PNS             PNS(X→Y)
               [0.0000, 0.6500]  width=0.6500  method=fiber

PN              PN (probability of necessity)
               [0.0000, 1.0000]  width=1.0000  method=fiber

PS              PS (probability of sufficiency)
               [0.0000, 1.0000]  width=1.0000  method=fiber



## Conditional interventional

$P(Y=1 \mid \mathrm{do}(X=1), W=0)$ on a three-variable order.


In [3]:
V3 = Variables.from_binary("W", "X", "Y")
P3 = {c: 0.125 for c in V3.cells()}
Qc = Query.interventional(outcome="Y", do={"X": 1}, value=1, condition={"W": 0})
rc = bound(V3, P3, Qc, Assumptions.order("W", "X", "Y"))
print(rc.summary(Qc))


query: P(Y=1 | do(X=1)), W=0
interval: [0.25, 0.75]
width: 0.5
method: closed/manski_binary_xy
tight: True
success: True
message: closed form


## Audit a bound

Structured report of atoms, faces, budgets, method, and tightness — useful
for notebooks and logging.


In [4]:
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
assumptions = a & Assumptions.mtr("Y", "X") & Assumptions.tan_msm("X", 2.0)
r = bound(V, P, Q, assumptions)
report = audit_bound(r, query=Q, assumptions=assumptions)
print(report)
print("faces:", report.faces)
print("budgets:", report.budgets)
print("atoms:", [atom["type"] for atom in report.atoms])


AuditReport(query='P(Y=1 | do(X=1))', method='msm_agg/msm_transport_lp_mtr', tight=True, faces=['MonotonicityAssumption'], budgets=['TanMSMAssumption'], interval=[0.55,0.775])
faces: ['MonotonicityAssumption']
budgets: ['TanMSMAssumption']
atoms: ['OrderAssumption', 'MonotonicityAssumption', 'TanMSMAssumption']


## Imbens–Manski-style confidence interval for the identified set

Treat the contingency table as multinomial with sample size `n`.


In [5]:
ci = bound_ci(V, P, Q, a, n_boot=50, n=500, seed=0)
print(ci)
print(
    f"point=[{ci.point_lower:.4f}, {ci.point_upper:.4f}]  "
    f"{100 * ci.level:.0f}% CI=[{ci.lower:.4f}, {ci.upper:.4f}]"
)


BoundCIResult(ci=[0.3097,0.8798], point=[0.35,0.85], level=0.95, method='imbens_manski')
point=[0.3500, 0.8500]  95% CI=[0.3097, 0.8798]
